In [2]:
# -*- coding: utf-8 -*-
"""
1D DAE benchmark with strict LHS-based evaluation timing and error computation.

Main purpose:
1. T_eval is measured on the same fixed LHS testing set used for e2/einf.
2. Error computation is performed after T_eval timing and is NOT included in T_eval.
3. T_eval is averaged over repeated reconstructions to reduce timing noise.
4. If LHS sample-index files from DAE-RAR already exist, this code reuses them
   only when they contain exactly NUM_SAMPLES unique indices.
5. Otherwise, it generates and saves exactly NUM_SAMPLES unique LHS-selected
   reference-grid indices.
6. The full loss curve is saved, but CPU transfer of the loss history is done
   after T_train timing using a 0-overhead list append strategy.
   Thus T_train does not include ANY synchronization overhead or kernel launches.

Important:
- This version guarantees that N_test = NUM_SAMPLES exactly.
- Therefore the paper statement "N_test = 5000 for the 1D problem" remains valid.
"""

import time
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.stats import qmc
from scipy.spatial import cKDTree


# =============================================================================
# Basic settings
# =============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

EPOCHS = 8000
N_F = 1000

X_MIN, X_MAX, T_FINAL = 0.0, 1.0, 0.3
H0_VALUE = 0.1

# LHS testing settings
NUM_SAMPLES = 5000
LHS_SEED = 1234

# Repeated timing settings for T_eval
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# Optional full-grid output for visualization only
SAVE_FULL_GRID = False
NX_FULL, NT_FULL = 201, 201

BASE_PATH = "."


# =============================================================================
# Network
# =============================================================================

class Net(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(layers[i], layers[i + 1]) for i in range(len(layers) - 1)]
        )
        self.activation = nn.Tanh()
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            init.xavier_normal_(m.weight)
            if m.bias is not None:
                init.zeros_(m.bias)

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        return self.layers[-1](x)


# =============================================================================
# Asymptotic components
# =============================================================================

CONST_600 = torch.tensor(600.0, dtype=torch.float32, device=DEVICE)
CONST_6 = torch.tensor(6.0, dtype=torch.float32, device=DEVICE)
CONST_145 = torch.tensor(145.0, dtype=torch.float32, device=DEVICE)


def phi_l_torch(x):
    return -torch.sqrt(
        CONST_600 + 6.0 * x**2 - 4.0 * x**3 + 3.0 * x**4
    ) / torch.sqrt(CONST_6)


def phi_r_torch(x):
    return torch.sqrt(
        CONST_145 + 6.0 * x**2 - 4.0 * x**3 + 3.0 * x**4
    ) / torch.sqrt(CONST_6)


def interface_residual(net, t):
    h = H0_VALUE + t * net(t)
    h_t = torch.autograd.grad(
        h.sum(),
        t,
        create_graph=True,
        retain_graph=True
    )[0]
    rhs = -0.5 * (phi_l_torch(h) + phi_r_torch(h))
    return h_t - rhs


def dae_reconstruct_gpu(net, x_eval, t_eval, mu):
    """
    Online DAE reconstruction.

    This is exactly the part timed by T_eval.
    It evaluates h_theta(t), phi^{(-/+)}(x), phi^{(-/+)}(h_theta(t)),
    and the Q_0-type layer correction.
    """
    with torch.no_grad():
        h = H0_VALUE + t_eval * net(t_eval)

        phi_l_x = phi_l_torch(x_eval)
        phi_r_x = phi_r_torch(x_eval)

        phi_l_h = phi_l_torch(h)
        phi_r_h = phi_r_torch(h)

        delta = phi_r_h - phi_l_h

        s_minus = torch.clamp((x_eval - h) * (-delta / (2.0 * mu)), -500.0, 500.0)
        s_plus = torch.clamp((x_eval - h) * (delta / (2.0 * mu)), -500.0, 500.0)

        u_left = phi_l_x + delta / (torch.exp(s_minus) + 1.0)
        u_right = phi_r_x - delta / (torch.exp(s_plus) + 1.0)

        return torch.where(x_eval <= h, u_left, u_right)


# =============================================================================
# LHS testing set and error computation
# =============================================================================

def true_u0_filename(mu):
    return f"1d_U0_true_mu{mu:.0e}_201_201_Mathematica.csv"


def lhs_index_filename(mu):
    mu_label = f"{mu:.0e}"
    return f"1d_LHS_sample_indices_mu{mu_label}.npy"


def lhs_points_filename(mu):
    mu_label = f"{mu:.0e}"
    return f"1d_LHS_test_points_mu{mu_label}.csv"


def load_true_solution_u0(mu):
    filename = true_u0_filename(mu)
    path = os.path.join(BASE_PATH, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find true solution file: {filename}")

    df = pd.read_csv(path)
    df = df.sort_values(by=["t", "x"]).reset_index(drop=True)
    return df, filename


def generate_lhs_indices(df_true, mu):
    """
    Generate exactly NUM_SAMPLES unique LHS-selected reference-grid indices.

    Procedure:
    1. Generate LHS points in the continuous (t,x) domain.
    2. Project them to nearest points of the reference grid by KDTree.
    3. If duplicates occur after nearest-neighbor projection, generate additional
       deterministic LHS batches until exactly NUM_SAMPLES unique indices are obtained.
    4. If this still fails, fill the remaining indices from unused grid points
       deterministically.

    This guarantees that the paper statement N_test = NUM_SAMPLES is exactly true.
    """
    total_points = len(df_true)

    if total_points < NUM_SAMPLES:
        raise ValueError(
            f"Reference grid has only {total_points} points, "
            f"which is smaller than NUM_SAMPLES={NUM_SAMPLES}."
        )

    t_min, t_max = df_true["t"].min(), df_true["t"].max()
    x_min, x_max = df_true["x"].min(), df_true["x"].max()

    all_points = df_true[["t", "x"]].values
    kdtree = cKDTree(all_points)

    selected = []
    used = set()

    batch_id = 0
    max_batches = 100

    while len(selected) < NUM_SAMPLES and batch_id < max_batches:
        sampler = qmc.LatinHypercube(d=2, seed=LHS_SEED + batch_id)

        lhs_sample = sampler.random(n=NUM_SAMPLES)
        lhs_sample_scaled = qmc.scale(
            lhs_sample,
            [t_min, x_min],
            [t_max, x_max]
        )

        _, candidate_indices = kdtree.query(lhs_sample_scaled)

        for idx in candidate_indices:
            idx = int(idx)

            if idx not in used:
                used.add(idx)
                selected.append(idx)

                if len(selected) == NUM_SAMPLES:
                    break

        batch_id += 1

    # Extremely unlikely fallback: fill remaining test points from unused grid points.
    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(
            np.arange(total_points),
            np.asarray(selected, dtype=int),
            assume_unique=False
        )

        if len(remaining) < NUM_SAMPLES - len(selected):
            raise RuntimeError(
                "Not enough remaining reference-grid points to complete "
                f"{NUM_SAMPLES} unique test points."
            )

        rng = np.random.default_rng(LHS_SEED)
        fill = rng.choice(
            remaining,
            size=NUM_SAMPLES - len(selected),
            replace=False
        )

        selected.extend([int(i) for i in fill])

    sample_indices = np.asarray(selected, dtype=int)

    # Final safety checks.
    if len(sample_indices) != NUM_SAMPLES:
        raise RuntimeError(
            f"Failed to generate exactly {NUM_SAMPLES} test points. "
            f"Got {len(sample_indices)}."
        )

    if len(np.unique(sample_indices)) != NUM_SAMPLES:
        raise RuntimeError("Generated LHS test indices are not unique.")

    np.save(lhs_index_filename(mu), sample_indices)

    return sample_indices


def build_or_load_lhs_test_set_from_true(mu):
    """
    Use exactly the same LHS indices as DAE-RAR if they already exist and are valid.
    Otherwise, generate exactly NUM_SAMPLES unique test indices and save them.

    The returned sample_indices are indices with respect to the sorted true dataframe.
    """
    df_true, filename = load_true_solution_u0(mu)

    index_file = lhs_index_filename(mu)

    if os.path.exists(index_file):
        sample_indices = np.load(index_file)

        valid_existing = (
            len(sample_indices) == NUM_SAMPLES
            and len(np.unique(sample_indices)) == NUM_SAMPLES
            and np.min(sample_indices) >= 0
            and np.max(sample_indices) < len(df_true)
        )

        if valid_existing:
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
        else:
            print(
                f"[mu={mu}] Existing LHS index file is invalid "
                f"(length={len(sample_indices)}, unique={len(np.unique(sample_indices))}). "
                f"Regenerating..."
            )

            sample_indices = generate_lhs_indices(df_true, mu)

            print(
                f"[mu={mu}] Regenerated and saved exactly "
                f"{len(sample_indices)} LHS indices to {index_file}."
            )
    else:
        sample_indices = generate_lhs_indices(df_true, mu)

        print(
            f"[mu={mu}] Generated and saved exactly "
            f"{len(sample_indices)} LHS indices to {index_file}."
        )

    # Final validation.
    if len(sample_indices) != NUM_SAMPLES or len(np.unique(sample_indices)) != NUM_SAMPLES:
        raise RuntimeError(
            f"LHS test set for mu={mu} is not exactly {NUM_SAMPLES} unique points."
        )

    t_lhs_np = df_true.iloc[sample_indices]["t"].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]["x"].values.reshape(-1, 1)
    true_lhs_np = df_true.iloc[sample_indices].iloc[:, 2].values.reshape(-1)

    t_lhs = torch.tensor(t_lhs_np, dtype=torch.float32, device=DEVICE)
    x_lhs = torch.tensor(x_lhs_np, dtype=torch.float32, device=DEVICE)

    df_test = pd.DataFrame({
        "t": t_lhs_np.reshape(-1),
        "x": x_lhs_np.reshape(-1),
        "u_true": true_lhs_np
    })
    df_test.to_csv(lhs_points_filename(mu), index=False)

    print(
        f"[mu={mu}] LHS test set built from {filename}: "
        f"N_test={len(sample_indices)}"
    )

    return {
        "df_true": df_true,
        "sample_indices": sample_indices,
        "t_lhs": t_lhs,
        "x_lhs": x_lhs,
        "t_lhs_np": t_lhs_np,
        "x_lhs_np": x_lhs_np,
        "true_lhs_np": true_lhs_np,
        "n_test": len(sample_indices)
    }


def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    e2 = np.linalg.norm(diff) / np.linalg.norm(true_u)
    einf = np.max(np.abs(diff))
    return e2, einf


def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return np.nanmean(arr), 0.0
    return np.nanmean(arr), np.nanstd(arr, ddof=1)


# =============================================================================
# Optional full-grid prediction
# =============================================================================

def save_full_grid_prediction(net, mu, seed):
    x_vals = np.linspace(X_MIN, X_MAX, NX_FULL)
    t_vals = np.linspace(0.0, T_FINAL, NT_FULL)
    T_grid, X_grid = np.meshgrid(t_vals, x_vals, indexing="ij")

    x_eval = torch.tensor(
        X_grid.reshape(-1, 1),
        dtype=torch.float32,
        device=DEVICE
    )
    t_eval = torch.tensor(
        T_grid.reshape(-1, 1),
        dtype=torch.float32,
        device=DEVICE
    )

    u_eval_tensor = dae_reconstruct_gpu(net, x_eval, t_eval, mu)
    u_pred = u_eval_tensor.detach().cpu().numpy().reshape(-1)

    df = pd.DataFrame({
        "x": X_grid.reshape(-1),
        "t": T_grid.reshape(-1),
        "u": u_pred
    })
    df.to_csv(f"1d_DAE_U0_predicted_mu{mu}_seed{seed}.csv", index=False)


# =============================================================================
# Main program
# =============================================================================

print("\n" + "=" * 80)
print("Starting 1D DAE benchmark with strict LHS-based T_eval and error")
print("=" * 80 + "\n")

print(f"Device: {DEVICE}")
print(f"LHS test points: N_test={NUM_SAMPLES}, LHS seed={LHS_SEED}")
print(f"T_eval warmup: {EVAL_WARMUP}, repeated timing: {EVAL_REPEAT}\n")

# Build or load LHS test sets before training.
# This makes DAE and DAE-RAR use exactly the same testing points.
lhs_data = {}
for mu in MU_LIST:
    lhs_data[mu] = build_or_load_lhs_test_set_from_true(mu)

metrics = {mu: [] for mu in MU_LIST}

for seed in SEEDS:
    print("\n" + "-" * 80)
    print(f"Running seed = {seed}")
    print("-" * 80)

    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fixed training points for DAE
    t_train = torch.empty(
        N_F,
        1,
        device=DEVICE
    ).uniform_(0.0, T_FINAL).requires_grad_(True)

    net = Net([1, 10, 10, 10, 10, 1]).to(DEVICE)
    optimizer = optim.Adam(net.parameters(), lr=1e-3)

    # Use a normal Python list to store 0D tensor references (0 overhead)
    loss_list = []

    net.train()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    train_start = time.perf_counter()

    for epoch in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        res = interface_residual(net, t_train)
        loss = torch.mean(res**2)

        loss.backward()
        optimizer.step()

        # Just append the reference, no data copy or CPU sync happens here!
        loss_list.append(loss.detach())

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    T_train = time.perf_counter() - train_start

    # Now that timing is completely finished, pull everything to the CPU
    # Now that timing is completely finished, pull the full loss history to CPU.
    loss_history = torch.stack(loss_list).detach().cpu().numpy().astype(np.float64)
    e_loss = float(loss_history[-1])

    T_train_per_iter_ms = T_train * 1e3 / EPOCHS
    T_train_per_iter_point_us = T_train * 1e6 / (EPOCHS * N_F)

    print(
        f" > trained: epochs={EPOCHS}, points={N_F}, "
        f"T_train={T_train:.2f}s, e_loss={e_loss:.3e}"
    )

    np.save(
        f"1d_DAE_loss_history_seed{seed}.npy",
        loss_history
    )

    net.eval()

    # -------------------------------------------------------------------------
    # T_eval and error for each mu.
    # T_eval is measured only on the fixed LHS test set.
    # Error is computed after timing and is not included in T_eval.
    # -------------------------------------------------------------------------

    for mu in MU_LIST:
        data_mu = lhs_data[mu]

        x_eval_lhs = data_mu["x_lhs"]
        t_eval_lhs = data_mu["t_lhs"]
        true_lhs_np = data_mu["true_lhs_np"]
        n_test = data_mu["n_test"]

        if n_test != NUM_SAMPLES:
            raise RuntimeError(
                f"N_test mismatch for mu={mu}: expected {NUM_SAMPLES}, got {n_test}."
            )

        # Warm-up on the same LHS testing set.
        with torch.no_grad():
            for _ in range(EVAL_WARMUP):
                _ = dae_reconstruct_gpu(net, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        # Repeated timing of online DAE reconstruction.
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        eval_start = time.perf_counter()

        with torch.no_grad():
            for _ in range(EVAL_REPEAT):
                _ = dae_reconstruct_gpu(net, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

        # Compute prediction once more for error.
        # This part is intentionally outside T_eval.
        with torch.no_grad():
            u_eval_tensor = dae_reconstruct_gpu(net, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        u_pred_lhs = u_eval_tensor.detach().cpu().numpy().reshape(-1)

        e2, einf = compute_error(true_lhs_np, u_pred_lhs)

        T_total = T_train + T_eval

        metrics[mu].append({
            "Seed": seed,
            "N_test": n_test,
            "e_loss": e_loss,
            "e2": e2,
            "einf": einf,
            "T_train": T_train,
            "T_eval": T_eval,
            "T_total": T_total,
            "T_train_per_iter_ms": T_train_per_iter_ms,
            "T_train_per_iter_point_us": T_train_per_iter_point_us,
            "total_trained_steps": EPOCHS,
            "total_point_steps": EPOCHS * N_F,
            "final_residual_points": N_F,
            "eval_warmup": EVAL_WARMUP,
            "eval_repeat": EVAL_REPEAT
        })

        print(
            f"    -> [mu={mu}] "
            f"N_test={n_test}, T_eval={T_eval:.6e}s, "
            f"e2={e2:.3e}, einf={einf:.3e}"
        )

        # Save LHS prediction on exactly the same testing set.
        mu_label = f"{mu:.0e}"
        df_lhs_pred = pd.DataFrame({
            "t": data_mu["t_lhs_np"].reshape(-1),
            "x": data_mu["x_lhs_np"].reshape(-1),
            "u": u_pred_lhs
        })
        df_lhs_pred.to_csv(
            f"1d_DAE_U0_predicted_LHS_mu{mu_label}_seed{seed}.csv",
            index=False
        )

        # Optional full-grid output for figures only.
        if SAVE_FULL_GRID:
            save_full_grid_prediction(net, mu, seed)


# =============================================================================
# Summary tables
# =============================================================================

print("\n" + "=" * 80)
print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
print("=" * 80 + "\n")

for mu in MU_LIST:
    dfm = pd.DataFrame(metrics[mu])

    mu_label = f"{mu:.0e}"
    dfm.to_csv(f"1d_DAE_mu{mu_label}_Metrics_Summary.csv", index=False)

    cols = [
        "e_loss",
        "e2",
        "einf",
        "T_train",
        "T_eval",
        "T_total",
        "T_train_per_iter_ms",
        "T_train_per_iter_point_us",
        "total_trained_steps",
        "total_point_steps",
        "final_residual_points",
        "N_test"
    ]

    stats = {c: mean_std(dfm[c].values) for c in cols}

    print(f"### Results for 1D DAE, mu={mu} [Mean \pm Sample Std] ###")
    print(f"N_test: {stats['N_test'][0]:.0f} \pm {stats['N_test'][1]:.0f}")
    print(f"e_loss: {stats['e_loss'][0]:.3e} \pm {stats['e_loss'][1]:.3e}")
    print(f"e_2: {stats['e2'][0]:.3e} \pm {stats['e2'][1]:.3e}")
    print(f"e_inf: {stats['einf'][0]:.3e} \pm {stats['einf'][1]:.3e}")
    print(f"T_train (s): {stats['T_train'][0]:.2f} \pm {stats['T_train'][1]:.2f}")
    print(f"T_eval (s): {stats['T_eval'][0]:.6e} \pm {stats['T_eval'][1]:.6e}")
    print(f"T_total (s): {stats['T_total'][0]:.2f} \pm {stats['T_total'][1]:.2f}")
    print(
        f"T_train/iter (ms): "
        f"{stats['T_train_per_iter_ms'][0]:.4f} \pm "
        f"{stats['T_train_per_iter_ms'][1]:.4f}"
    )
    print(
        f"T_train/(iter*pt) (us): "
        f"{stats['T_train_per_iter_point_us'][0]:.4f} \pm "
        f"{stats['T_train_per_iter_point_us'][1]:.4f}"
    )
    print(
        f"Optimization steps: "
        f"{stats['total_trained_steps'][0]:.1f} \pm "
        f"{stats['total_trained_steps'][1]:.1f}"
    )
    print(
        f"Point-iterations: "
        f"{stats['total_point_steps'][0]:.1f} \pm "
        f"{stats['total_point_steps'][1]:.1f}"
    )
    print(
        f"Final residual points: "
        f"{stats['final_residual_points'][0]:.1f} \pm "
        f"{stats['final_residual_points'][1]:.1f}"
    )
    print()


Starting 1D DAE benchmark with strict LHS-based T_eval and error

Device: cuda
LHS test points: N_test=5000, LHS seed=1234
T_eval warmup: 20, repeated timing: 200

[mu=0.01] Loaded valid LHS indices from 1d_LHS_sample_indices_mu1e-02.npy.
[mu=0.01] LHS test set built from 1d_U0_true_mu1e-02_201_201_Mathematica.csv: N_test=5000
[mu=0.001] Loaded valid LHS indices from 1d_LHS_sample_indices_mu1e-03.npy.
[mu=0.001] LHS test set built from 1d_U0_true_mu1e-03_201_201_Mathematica.csv: N_test=5000
[mu=0.0001] Loaded valid LHS indices from 1d_LHS_sample_indices_mu1e-04.npy.
[mu=0.0001] LHS test set built from 1d_U0_true_mu1e-04_201_201_Mathematica.csv: N_test=5000

--------------------------------------------------------------------------------
Running seed = 33
--------------------------------------------------------------------------------
 > trained: epochs=8000, points=1000, T_train=63.10s, e_loss=2.520e-07
    -> [mu=0.01] N_test=5000, T_eval=9.665725e-04s, e2=4.917e-04, einf=8.207e-02
 